# Neural Amp Modeler — Temporal effects (CUDA / Colab / Kaggle)

**Before you run:** enable a **GPU** (Colab: *Runtime → Change runtime type → GPU*. Kaggle: *Settings → Accelerator → GPU*).

This notebook trains the **temporal** (delay/reverb-style) model from this fork. It does **not** use the standard Colab trainer `nam.train.colab.run()` — that path targets the classic WaveNet-style amp model only.

Exports are still plugin-compatible **`.nam`** files from the same export path as local `train.py`.

## Step 1: Get data

* **Dry signal:** `input.wav` (the same reamp / DI file you used for the wet capture).
* **Wet capture:** `output.wav` (reamp through your time-based effect). Prefer **48 kHz**, **24-bit**, **mono** (same as the [official NAM Colab tutorial](https://neural-amp-modeler.readthedocs.io/en/stable/tutorials/colab.html)). This fork also supports **stereo** pairs if channel counts match.
* **Upload:** Colab — use the folder panel and upload both files (often under `/content`). Kaggle — add a **Dataset** containing `input.wav` / `output.wav` and reference paths under `/kaggle/input/...`, or copy them into `/kaggle/working`.

Set `INPUT_WAV` and `OUTPUT_WAV` in the next code section to match where those files landed.

### Faster runs (optional)

* **`steps`** — wall time scales about linearly with `max_steps`.
* **`precision`** — use **`16-mixed`** on CUDA for a large throughput win; keep **`32-true`** if you hit instability.
* **`mrstft_weight`** — set to **`0`** to skip MR-STFT in the *training* loss for quick experiments (often much faster; quality may suffer).
* **`val_check_interval`** — validation is expensive, but PyTorch Lightning requires this to be **≤ train batches per epoch** (≈ `ceil(epoch_steps / batch_size)`). Example: `epoch_steps=2000`, `batch_size=8` → at most **250**. Larger values need a higher `epoch_steps` (e.g. `8000`+ for interval `1000`).
* **`preview_every` / `checkpoint_every`** — larger values mean less disk I/O (preview WAVs and checkpoints).
* **`batch_size`** — increase while VRAM allows; stereo runs effectively double batch width after channel flattening.
* **`context` / `target`** — shorter windows speed up each step but cap how much reverb tail the model sees at once (quality tradeoff).
* **`hidden_size` / `local_layers`** — smaller / fewer layers train faster with less capacity.

`epoch_steps` changes how many batches define one DataLoader epoch (and scales default validation length). It does **not** replace lowering `steps` if you want fewer total optimizer updates.

### Kaggle notes

* Write outputs under **`/kaggle/working`** so they persist for download.
* Put training WAVs in an **Input Dataset** and point `INPUT_WAV` / `OUTPUT_WAV` at `/kaggle/input/<dataset>/...`.
* If DataLoader workers crash, set **`num_workers`** to **`0`** or **`2`**.

In [ ]:
# Install this fork (run once per session). Uses subprocess so it works on Colab and Kaggle.
import importlib.util
import subprocess
import sys


def _pip_install(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


INSTALL_MODE = "git"  #@param ["git", "local_editable"] {type:"string"}
# Full git HTTPS URL to the repo root (must contain this fork's pyproject.toml), e.g. https://github.com/you/neural-amp-modeler.git
GIT_REPO_URL = "https://github.com/robjlyons/nam-time-temporal.git"  #@param {type:"string"}
GIT_BRANCH = "main"  #@param {type:"string"}
# After unzipping or cloning the repo into the runtime, point here for editable install:
LOCAL_PACKAGE_DIR = "/content/neural-amp-modeler"  #@param {type:"string"}

if importlib.util.find_spec("nam") is None or importlib.util.find_spec("nam.training") is None:
    if INSTALL_MODE == "git":
        url = GIT_REPO_URL.strip().rstrip("/")
        if not url.startswith("git+"):
            url = "git+" + url
        if "@" not in url.split("#", 1)[0]:
            url = f"{url}@{GIT_BRANCH}"
        print("Installing from", url)
        _pip_install(url)
    else:
        print("Editable install from", LOCAL_PACKAGE_DIR)
        _pip_install("-e", LOCAL_PACKAGE_DIR)
else:
    print("nam already importable; skipping install.")

# If the next cell still cannot import nam.training, use Runtime → Restart session (Colab) and re-run this cell.
print("Install cell done.")

In [ ]:
from pathlib import Path

from nam.training.config import TemporalTrainingConfig
from nam.training.engine import train_temporal_model

%load_ext tensorboard

# --- Paths (edit for your environment) ---
INPUT_WAV = "/content/input.wav"  #@param {type:"string"}
OUTPUT_WAV = "/content/output.wav"  #@param {type:"string"}
OUTDIR = "/content/nam_temporal_out"  #@param {type:"string"}
RESUME_CKPT = ""  #@param {type:"string"}

# --- Training ---
steps = 20000  #@param {type:"integer"}
batch_size = 8  #@param {type:"integer"}
context = 8192  #@param {type:"integer"}
target = 8192  #@param {type:"integer"}
overlap = 1024  #@param {type:"integer"}
epoch_steps = 2000  #@param {type:"integer"}
hidden_size = 48  #@param {type:"integer"}
local_layers = 2  #@param {type:"integer"}
learning_rate = 3e-4  #@param {type:"number"}
val_check_interval = 250  #@param {type:"integer"}
checkpoint_every = 500  #@param {type:"integer"}
preview_every = 1000  #@param {type:"integer"}
log_every = 50  #@param {type:"integer"}
num_workers = 2  #@param {type:"integer"}
prefetch_factor = 2  #@param {type:"integer"}
precision = "16-mixed"  #@param ["32-true", "16-mixed"] {type:"string"}
mrstft_weight = 0.0002  #@param {type:"number"}
device = "auto"  #@param ["auto", "cpu", "gpu"] {type:"string"}
force_mono = False  #@param {type:"boolean"}

outdir = Path(OUTDIR)
outdir.mkdir(parents=True, exist_ok=True)

from IPython import get_ipython

_ipy = get_ipython()
if _ipy is not None:
    _ipy.run_line_magic("tensorboard", f"--logdir {outdir}")

cfg = TemporalTrainingConfig(
    input_wav=Path(INPUT_WAV),
    output_wav=Path(OUTPUT_WAV),
    outdir=outdir,
    max_steps=int(steps),
    batch_size=int(batch_size),
    num_workers=int(num_workers),
    persistent_workers=int(num_workers) > 0,
    prefetch_factor=int(prefetch_factor),
    context_samples=int(context),
    target_samples=int(target),
    overlap_samples=int(overlap),
    epoch_steps=int(epoch_steps),
    hidden_size=int(hidden_size),
    local_layers=int(local_layers),
    learning_rate=float(learning_rate),
    val_check_interval=int(val_check_interval),
    checkpoint_every_n_steps=int(checkpoint_every),
    preview_every_n_steps=int(preview_every),
    log_every_n_steps=int(log_every),
    precision=str(precision),
    mrstft_weight=float(mrstft_weight) if float(mrstft_weight) > 0 else None,
    resume=Path(RESUME_CKPT) if str(RESUME_CKPT).strip() else None,
    device=str(device),
    force_mono=bool(force_mono),
)

train_temporal_model(cfg)
print("Training finished. Export:", outdir / "model.nam")

## Step 3: Check results and download

* **`model.nam`** — in `OUTDIR` (same directory Lightning uses as `default_root_dir`). Refresh the file browser if you do not see it.
* **`checkpoints/`** — Lightning checkpoints (`step...ckpt`).
* **`previews/`** — periodic WAV comparisons when previews are enabled.
* TensorBoard — charts are logged under `OUTDIR` (versioned `lightning_logs` subfolders).

### Enjoy

Load `model.nam` in the Neural Amp Modeler plugin like any other NAM model.